# Gumbell Softmax

---
## definition
- A way to sample from a categorical distribution in differentiable way using reparametrization

---
## gumbel noise
- it allows to simulate `argmax` sampling process, which is equivalent to sampling from categorical distribution
- `gumbel_sample = -log(-log(U))` where `U ~ Uniform(0, 1)

---
## Reparameterization Trick

instead of doing

```python
y = one_hot(argmax(logits + gumble_noise)) # not differentiable
```

do this operation

```python
y_i = softmax((logits + gumple_noise)/temperature) 
# as temperature goes to zero output becomes one_hot and as temperature goes to infinity we get one_hot
```

---
# Example

Suppose having a distribution to pick one of the balls of 3 different colors is given as
- red = 0.1
- green = 0.3
- blue = 0.6

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn

# logits from prediction

logits = torch.tensor([0.2, 0.5, 2.0])
print(f"logits: {logits}")

logits: tensor([0.2000, 0.5000, 2.0000])


In [ ]:
# simple softmax
smax_result = F.softmax(logits, dim=0)
print(f"softmax output of logits: {smax_result}")

# gumbel softmax
# running everytime would give a different result
U = torch.rand_like(logits)
gumble_noise = -torch.log(-torch.log(U))
tau = 0.5  # temperature
gumble_softmax_result = F.softmax((logits + gumble_noise) / tau, dim=0)
print(f"gumbel softmax output of logits: {gumble_softmax_result}")

softmax output of logits: tensor([0.1191, 0.1607, 0.7202])
gumbel softmax output of logits: tensor([0.0182, 0.0046, 0.9772])


In [22]:
import torch
import torch.nn.functional as F
import torch.nn as nn

def gumbel_softmax(logits, temperature=1.0, hard=False):
    # Sample Gumbel noise
    noise = -torch.log(-torch.log(torch.rand_like(logits) + 1e-10) + 1e-10)
    y = F.softmax((logits + noise) / temperature, dim=-1)
    
    if hard:
        # Convert to one-hot
        y_hard = torch.zeros_like(y)
        y_hard.scatter_(1, y.argmax(dim=1, keepdim=True), 1.0)
        y = (y_hard - y).detach() + y  # Straight-through estimator
    return y


In [44]:
logits = torch.randn(50, 100)

In [45]:
gumbel_softmax(logits, 1.0, hard=True)

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

# Implementation1

- The idea is each layer learns a mask which will sample some neurons from each layer.
- Its based on the idea we talked about that the model does not have to look at every concept that it has inside of it it can look at less concepts by this sampling

## problems
- the mask is learned based on overall data, I would like it to be input dependent will think of another implementation


In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn

class GumbelSamplingLayer(nn.Module):
    def __init__(self, in_dim, full_out_dim, sample_dim, temperature=1.0):
        super().__init__()
        self.linear = nn.Linear(in_dim, full_out_dim)
        self.logits = nn.Parameter(torch.randn(sample_dim, full_out_dim))  # learnable selector
        self.sample_dim = sample_dim
        self.temperature = temperature

    def forward(self, x):
        # x: (batch_size, in_dim)
        full_out = self.linear(x)  # (batch_size, full_out_dim)

        # Sample selection mask: (sample_dim, full_out_dim)
        mask_weights = gumbel_softmax(self.logits, temperature=self.temperature, hard=True)

        # Weighted sum: project from full_out_dim -> sample_dim
        # Output shape: (batch_size, sample_dim)
        sampled_out = torch.matmul(mask_weights, full_out.T).T
        return sampled_out


# Implementation2

- this is another approach. 
- I learn the logits of each choice and then sample from there but the parameter count is too large

In [23]:
import torch
import torch.nn.functional as F
import torch.nn as nn

class InputDependentGumbelSamplingLayer(nn.Module):
    def __init__(self, in_dim, full_out_dim, sample_dim, temperature=1.0, hidden_dim=64):
        super().__init__()
        self.linear = nn.Linear(in_dim, full_out_dim)
        self.sample_dim = sample_dim
        self.full_out_dim = full_out_dim
        self.temperature = temperature

        # Project input into logits for sampling mask: (batch_size, sample_dim, full_out_dim)
        self.mask_mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            # nn.Linear(hidden_dim, sample_dim * full_out_dim)
            nn.Linear(hidden_dim, sample_dim * full_out_dim)
        )

    def forward(self, x):
        batch_size = x.size(0)
        full_out = self.linear(x)  # (batch_size, full_out_dim)

        # Create per-sample logits for mask
        mask_logits = self.mask_mlp(x)  # (batch_size, sample_dim * full_out_dim)
        mask_logits = mask_logits.view(batch_size, self.sample_dim, self.full_out_dim)

        # Apply Gumbel-Softmax per sample
        mask = gumbel_softmax(mask_logits, temperature=self.temperature, hard=True)  # (batch, sample_dim, full_out_dim)

        # Mask full_out: we want (batch, sample_dim)
        # full_out: (batch, full_out_dim)
        # → need to apply each sample's mask to its output row

        sampled = torch.bmm(mask, full_out.unsqueeze(2)).squeeze(2)  # (batch, sample_dim)

        return sampled


# Implementation3
- factorize the sampling matrix

In [47]:
class FactorizedInputDependentGumbelSamplingLayer(nn.Module):
    def __init__(self, in_dim, full_out_dim, sample_dim, temperature=1.0, hidden_dim=64):
        super().__init__()
        self.linear = nn.Linear(in_dim, full_out_dim)
        self.sample_dim = sample_dim
        self.full_out_dim = full_out_dim
        self.temperature = temperature

        # Factorized routing
        self.row_router = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, sample_dim)  # (B, sample_dim)
        )

        self.col_router = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, full_out_dim)  # (B, full_out_dim)
        )

    def forward(self, x):
        full_out = self.linear(x)  # (B, full_out_dim)

        # Routing logits
        row_logits = self.row_router(x)           # (B, sample_dim)
        col_logits = self.col_router(x)           # (B, full_out_dim)

        # Outer product to get mask logits: (B, sample_dim, full_out_dim)
        mask_logits = torch.einsum('bi,bj->bij', row_logits, col_logits)

        # Gumbel-softmax over full_out_dim for each output slot
        mask = gumbel_softmax(mask_logits, temperature=self.temperature, hard=True)  # (B, sample_dim, full_out_dim)

        # Apply mask to full_out: (B, sample_dim) ← (B, S, F) × (B, F, 1)
        sampled = torch.bmm(mask, full_out.unsqueeze(2)).squeeze(2)  # (B, sample_dim)

        return sampled